# Run pipeline on CFPB data

Thin wrapper around `src.build` — set a CSV path, run the cell, then inspect results.

Needs a CFPB export with a **Consumer complaint narrative** column (Narratives Archive Full Records). Terminal alternative: `python -m src.build --csv /path/to/file.csv`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Point this at any CFPB CSV with narratives, or leave as the default path.
CSV_PATH = ROOT / "data" / "raw" / "complaints.csv"
# CSV_PATH = Path("/path/to/another-export.csv")

print("CSV:", CSV_PATH)
print("exists:", CSV_PATH.exists())

In [ ]:
from src.build import build

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CSV_PATH}. Copy a Narratives Archive CSV there, "
        "or set CSV_PATH to another file in the cell above."
    )

build(csv_path=CSV_PATH)

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import Image, display

DB = ROOT / "results" / "signals.db"
FIG = ROOT / "results" / "figures"

con = sqlite3.connect(DB)
n = pd.read_sql("SELECT COUNT(*) AS n FROM fct_interaction", con).iloc[0, 0]
print(f"rows in fct_interaction: {n:,}")
display(pd.read_sql(
    """
    SELECT primary_pain_point, switching_intent, lead_signal_score, amount_usd_max
    FROM fct_interaction
    ORDER BY lead_signal_score DESC
    LIMIT 10
    """,
    con,
))

In [ ]:
for name in [
    "switching_by_pain_point.png",
    "lead_signal_distribution.png",
    "hot_leads_by_pain_point.png",
]:
    path = FIG / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"missing {name}")

con.close()